In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation, FFMpegWriter
import os
import time


t_i = 0
t_f = 10
t_steps = 200

x_i = -20
x_f = 180
x_steps = 1000

t = np.linspace(t_i, t_f, t_steps)
x = np.linspace(x_i, x_f, x_steps)


def gaussian_distribution(val, peak, width):
    return np.exp(-(val-peak)**2/(width)**2 * 1/2)


N_k = 1000
k_0 = 2
dk = 5
k = np.linspace(k_0 - dk/2, k_0 + dk/2, N_k)

c = 10
w_0 = 10
dw = 10
w = np.linspace(w_0 - dw/2, w_0 + dw/2, N_k)

wave = np.zeros_like(x)

def dispersion(k):
    return c* (2*k_0) * np.sqrt(np.abs(k))*np.sign(k)
    
def pulse(x, t):
    wave = np.zeros_like(x)
    for i in range(N_k):
        wave += 1/N_f * gaussian_distribution(k[i], k_0, df/4) * np.sin(k[i]*x - dispersion(k[i])*t)
    return wave

def create_animation(
        x,
        num_frames=None,
        output_file="wave.mp4",
        fps=30,
        bitrate=1800,
        figsize=(8,4),
        interval=20):

    if num_frames is None:
        num_frames = len(t)

    # Create figure
    fig, ax = plt.subplots(figsize=figsize)

    # Initial curve
    y0 = pulse(x, t[0])
    line, = ax.plot(x, y0, lw=2)

    ax.set_xlabel("x")
    ax.set_ylabel("Amplitude")
    ax.set_title(fr"$k_0$ = {k_0}, $\Delta k$ = {dk}; t = {t[0]:.3f}")

    # Fix axes limits
    ymax = 1.1 * np.max(np.abs(y0))
    ax.set_xlim(np.min(x), np.max(x))
    ax.set_ylim(-ymax, ymax)

    fig.tight_layout(pad=0.2)

    # Update function
    def update(frame):
        current_y = pulse(x, t[frame])

        line.set_ydata(current_y)
        ax.set_title(fr"$k_0$ = {k_0}, $\Delta k$ = {dk}; t = {t[frame]:.3f}")

        return (line,)

    print("Creating animation...")

    ani = FuncAnimation(
        fig,
        update,
        frames=num_frames,
        interval=interval,
        blit=True
    )

    print(f"Saving animation to: {output_file}")

    writer = FFMpegWriter(
        fps=fps,
        bitrate=bitrate
    )

    ani.save(output_file, writer=writer)

    plt.show()

    return ani


In [ ]:
ani = create_animation(
    x,
    output_file=f"wave_packet_k({k_0}_{dk})_w({w_0}_{dw}).mp4",
    fps=30
)